# R03 — Confirmation across factual anchors, verification rates, and a correct rerun of the baseline-retrieval confirmation

**GPU required (≈ 2–5 h per dataset at the defaults; `REVIEW_SMOKE=1` = plumbing check).** Needs `MVTEC_ROOT` and `PDT_ROOT`.
Resumable (cache in `review/outputs/cache/`). Addresses **R2 "one anchor drives the main negative finding … increase the
number of confirmation candidates and include those collected with at least two more anchors; report how the verification
rate changes and how the original anchor was picked"**, and repairs the §4.9 / Table 6 evidence that `R00` showed to be invalid for MVTec.

### Blocks
| block | what |
|---|---|
| A | **How was the primary anchor picked?** Is it the best logged trial or a random one? Where does it sit in the log distribution? |
| B | **Multi-anchor confirmation.** For each of the five anchors (p50, p70, p85, p95, best) of `*_multifactual_candidates.csv` (DiCE and LogTopK, K = 8 each) the top `N_PER_CELL` candidates *by surrogate* are re-trained, i.e. exactly the candidates an operator would be shown. |
| C | **Verification rate** per anchor and method: share of candidates whose re-measured validation F1 meets the query objective (F1 ≥ 0.75) and share within ±0.05 of the surrogate prediction. |
| D | **Does the gap track the anchor?** error vs anchor percentile / anchor value / distance to the factual. |
| E | **Correct rerun of the baseline confirmation** (`baseline_confirm_candidates_*.csv`, 6 candidates per dataset, DiCE / LogTopK / NearestFeasible) using the same code path as A–D. The committed `baseline_confirmation_mvtec.csv` was produced on the wrong data (R00: implied test prevalence 0.339 vs 0.702, AUROC 0.975). |

Protocol: MVTec = one PatchCore + one review-budget threshold per category (the protocol of the surrogate’s target); Proditec = one model.
The pooled-MVTec variant of the same models is also reported so that the old and new numbers can be linked.

In [ ]:
# ============================================================
# C0) Common setup (identical in every review notebook)
# ============================================================
from pathlib import Path
import json, re, sys, warnings
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 220)
try:
    display
except NameError:
    display = print

SEED = 42
np.random.seed(SEED)

plt.rcParams.update({
    "font.family": "serif", "font.size": 8, "axes.labelsize": 8,
    "axes.titlesize": 8, "xtick.labelsize": 7, "ytick.labelsize": 7,
    "legend.fontsize": 7, "figure.dpi": 150, "savefig.dpi": 300,
    "axes.spines.top": False, "axes.spines.right": False,
})
SINGLE_COL = (3.35, 2.35)
WIDE_COL = (6.85, 2.65)

# ------------------------------------------------------------
# Robust repo-root detection (CWD, its parents, then rglob)
# ------------------------------------------------------------
CWD = Path.cwd().resolve()


def _is_repo(p: Path) -> bool:
    return (p / "mvtec" / "outputs_patchcore").is_dir() and (p / "proditec" / "outputs_patchcore").is_dir()


REPO = next((p for p in [CWD, *CWD.parents] if _is_repo(p)), None)
if REPO is None:
    for base in [CWD, CWD.parent]:
        for hit in base.rglob("outputs_patchcore"):
            if _is_repo(hit.parent.parent):
                REPO = hit.parent.parent
                break
        if REPO:
            break
if REPO is None:
    raise FileNotFoundError("Could not locate the repository root (needs mvtec/ and proditec/). "
                            "Launch Jupyter from the repo or from review/.")
REVIEW = REPO / "review"
OUT = REVIEW / "outputs"
OUT_TABLES, OUT_FIGS = OUT / "tables", OUT / "figures"
for d in (OUT_TABLES, OUT_FIGS):
    d.mkdir(parents=True, exist_ok=True)

DS = {
    "mvtec": dict(root=REPO / "mvtec", pre="mvtec",
                  data="surrogate_ready_dataset/mvtec_4class_patchcore_surrogate_dataset.csv"),
    "proditec": dict(root=REPO / "proditec", pre="proditec",
                     data="surrogate_ready_dataset/patchcore_surrogate_dataset_xgb.csv"),
}
for k, v in DS.items():
    r = v["root"]
    v["art"] = r / "surrogate_pkl_cfs_metadata"
    v["cfs"] = v["art"] / f"{k}_counterfactuals" / f"{k}_all_methods_cfs.csv"
    v["eval"] = v["art"] / f"{k}_counterfactuals_evaluation" / f"{k}_all_methods_cfs_patchcore_evaluation.csv"
    v["rev"] = v["art"] / f"{k}_revision_analysis"
    v["decomp"] = r / "revision_outputs" / "tables" / f"decomposition_per_candidate_{k}.csv"
    v["model"] = v["art"] / f"{k}_surrogate_catboost.cbm"
    v["meta"] = v["art"] / f"{k}_surrogate_metadata.json"
    v["dataset"] = r / v["data"]
    v["confirm"] = r / "revision_outputs" / "tables" / f"baseline_confirmation_{k}.csv"
    v["confirm_cands"] = r / "revision_outputs" / "tables" / f"baseline_confirm_candidates_{k}.csv"
print("REPO  :", REPO)
print("OUT   :", OUT)


def save_table(table: pd.DataFrame, stem: str, ff="%.4f"):
    p = OUT_TABLES / f"{stem}.csv"
    table.to_csv(p, index=False)
    try:
        table.to_latex(OUT_TABLES / f"{stem}.tex", index=False, escape=True, float_format=ff)
    except Exception as e:
        print("  (tex export skipped:", e, ")")
    print("Saved table:", p.relative_to(REPO))
    return p


def save_fig(fig, stem: str):
    for ext in ("pdf", "png"):
        fig.savefig(OUT_FIGS / f"{stem}.{ext}", bbox_inches="tight")
    plt.close(fig)
    print("Saved figure:", (OUT_FIGS / f"{stem}.pdf").relative_to(REPO))


def merge_numbers(section: str, values: dict):
    """Append to outputs/frozen_numbers.json (one section per notebook)."""
    p = OUT / "frozen_numbers.json"
    cur = json.loads(p.read_text()) if p.exists() else {}
    cur[section] = json.loads(json.dumps(values, default=float))
    p.write_text(json.dumps(cur, indent=2, sort_keys=True))
    print("Updated", p.relative_to(REPO), "->", section)

In [ ]:
# ============================================================
# C1) GPU / dataset configuration  <-- EDIT THIS CELL
# ============================================================
import os, time, pickle

# Local copies of the datasets (the split CSVs hold absolute paths of the original server and are
# re-rooted onto these folders).  Environment variables take precedence.
#   MVTEC_ROOT : folder that contains bottle/ cable/ capsule/ hazelnut/ (MVTec AD)
#   PDT_ROOT   : folder that contains train/ and test/ (Proditec public dataset; proprietary)
MVTEC_ROOT = os.environ.get("MVTEC_ROOT", "")      # e.g. "/data/mvtec_anomaly_detection"
PDT_ROOT = os.environ.get("PDT_ROOT", "")          # e.g. "/data/pdt/dataset_public"
DEVICE_NAME = os.environ.get("REVIEW_DEVICE", "")  # "" = auto (cuda if available), or "cuda:1" / "cpu"
SMOKE = os.environ.get("REVIEW_SMOKE", "0") == "1" # 1 = tiny run (2-3 items) to check the plumbing
NUM_WORKERS = int(os.environ.get("REVIEW_WORKERS", "2"))

sys.path.insert(0, str(REVIEW))
import torch
import patchcore_review_lib as L
L.NUM_WORKERS = NUM_WORKERS
L.set_device(DEVICE_NAME or None)
torch.manual_seed(SEED)
if SMOKE:   # plumbing runs never touch the real outputs / cache
    OUT = OUT / "smoke"
    OUT_TABLES, OUT_FIGS = OUT / "tables", OUT / "figures"
    for _d in (OUT_TABLES, OUT_FIGS):
        _d.mkdir(parents=True, exist_ok=True)
CACHE = OUT / "cache"
CACHE.mkdir(exist_ok=True)
print("device:", L.DEVICE, "| smoke:", SMOKE)
print("MVTEC_ROOT:", MVTEC_ROOT or "(unset)", "| PDT_ROOT:", PDT_ROOT or "(unset)")


def need_images(df: pd.DataFrame, what: str):
    rate = L.path_exist_rate(df)
    print(f"{what}: {len(df)} images, path_exists_rate={rate:.3f}")
    if rate < 1.0:
        raise FileNotFoundError(f"{what}: some images are missing. Set MVTEC_ROOT / PDT_ROOT "
                                "(see the configuration cell).")


def load_data(ds: str):
    """ds in {'mvtec' (per-category dict), 'mvtec_pooled' (one pooled frame set), 'proditec'}."""
    if ds.startswith("mvtec"):
        cd = L.load_mvtec_class_data(REPO / "mvtec" / "mvtec_splits", MVTEC_ROOT or None)
        if SMOKE:
            cd = {c: L.smoke_subsample(v, 14) for c, v in cd.items()}
        data = L.pool_class_data(cd) if ds == "mvtec_pooled" else cd
        if ds == "mvtec":
            for c, v in cd.items():
                need_images(pd.concat([v["train"], v["val"], v["final"]]), f"MVTec/{c}")
        else:
            need_images(pd.concat([data["train"], data["val"], data["final"]]), "MVTec pooled")
        return data
    data = L.load_proditec_data(REPO / "proditec" / "patchcore_splits", PDT_ROOT or None)
    if SMOKE:
        data = L.smoke_subsample(data, 40)
    need_images(pd.concat([data["train"], data["val"], data["final"]]), "Proditec")
    return data

In [ ]:
# ============================================================
# R3.1) Configuration and inputs
# ============================================================
import hashlib
from catboost import CatBoostRegressor
from scipy.stats import spearmanr

N_PER_CELL = int(os.environ.get("REVIEW_N_PER_CELL", "3"))        # candidates per anchor x method (top by surrogate)
N_SEEDS = int(os.environ.get("REVIEW_SEEDS", "1"))
TARGET_MIN = 0.75                                                 # objective of the multi-anchor sweep (paper §4.8)
TOL = 0.05                                                        # 'close to the surrogate'
DATASETS = ["mvtec", "proditec"]
if SMOKE:
    N_PER_CELL, N_SEEDS = 1, 1
    ANCHOR_FILTER = ["best", "p50"]
else:
    ANCHOR_FILTER = None
SEEDS = tuple(SEED + i for i in range(N_SEEDS))

SURR, MULTI, LOG, BASE = {}, {}, {}, {}
for ds in DATASETS:
    s = DS[ds]
    meta = json.loads(s["meta"].read_text()); mdl = CatBoostRegressor(); mdl.load_model(str(s["model"]))
    SURR[ds] = (mdl, meta)
    LOG[ds] = pd.read_csv(s["dataset"])
    MULTI[ds] = pd.read_csv(s["rev"] / f"{ds}_multifactual_candidates.csv")
    BASE[ds] = pd.read_csv(s["confirm_cands"])
    print(f"{ds:9s} log rows {len(LOG[ds])} | multifactual rows {len(MULTI[ds])} | baseline-confirm rows {len(BASE[ds])}")


def surrogate_predict(ds, frame):
    mdl, meta = SURR[ds]
    X = pd.DataFrame({f: frame[f].values if f in frame else frame[f.replace("params_", "")].values for f in meta["features"]})
    for c in meta["categorical_features"]:
        X[c] = X[c].astype(str).str.replace(r"\.0$", "", regex=True)
    return np.asarray(mdl.predict(X[list(mdl.feature_names_)]), float)

In [ ]:
# ============================================================
# R3.2) Block A - how was the primary anchor picked?
# ============================================================
rows = []
for ds in DATASETS:
    meta = json.loads((DS[ds]["art"] / f"{ds}_counterfactuals" / f"{ds}_all_methods_cfs_metadata.json").read_text())
    log = LOG[ds]; fi = int(meta["factual_index"])
    v = log["value"]
    rows.append(dict(dataset=ds, primary_anchor_index=fi, observed_F1=float(meta["factual_value_observed"]),
                     surrogate_F1_at_anchor=float(meta["factual_value_surrogate"]),
                     log_max=float(v.max()), is_best_logged_trial=bool(abs(v.loc[fi] - v.max()) < 1e-12),
                     percentile_in_log=float((v < v.loc[fi]).mean() * 100), n_log=len(log),
                     anchor_review_budget=float(log.loc[fi, "params_soft_review_budget"]),
                     anchor_train_fraction=float(log.loc[fi, "params_soft_train_fraction"])))
A_tab = pd.DataFrame(rows); display(A_tab.round(4)); save_table(A_tab, "R03_primary_anchor_provenance")
multi_anchor = {ds: MULTI[ds].groupby("anchor").anchor_index.first().to_dict() for ds in DATASETS}
print("Multi-anchor indices:", multi_anchor)
print("\nIf `is_best_logged_trial` is True the primary query started from the single best logged configuration, i.e. the extreme upper tail "
      "of the log, the regime in which surrogate over-optimism is largest. The multi-anchor sweep therefore tests a *less* extreme start.")

In [ ]:
# ============================================================
# R3.3) Select the candidates to confirm (top by surrogate per anchor x method)
# ============================================================
sel = {}
for ds in DATASETS:
    m = MULTI[ds].reset_index().rename(columns={"index": "row"})
    if ANCHOR_FILTER:
        m = m[m.anchor.isin(ANCHOR_FILTER)]
    m = m.sort_values(["anchor", "method", "surrogate_pred"], ascending=[True, True, False])
    top = m.groupby(["anchor", "method"]).head(N_PER_CELL).copy()
    top["cid"] = [f"m{int(r)}" for r in top.row]
    top["anchor_value"] = top.anchor_index.map(LOG[ds]["value"])
    top["anchor_surrogate"] = surrogate_predict(ds, LOG[ds].loc[top.anchor_index.values].reset_index(drop=True))
    top["anchor_pctile"] = top.anchor_index.map(lambda i: (LOG[ds]["value"] < LOG[ds]["value"].loc[i]).mean() * 100)
    sel[ds] = top.reset_index(drop=True)
    print(f"{ds}: {len(top)} candidates = {top.anchor.nunique()} anchors x {top.method.nunique()} methods x <= {N_PER_CELL}")

In [ ]:
# ============================================================
# R3.4) Run: multi-anchor candidates (block B) and baseline-confirm candidates (block E)
# ============================================================
RES = {}
for ds in DATASETS:
    data = load_data(ds)
    top = sel[ds]
    print(f"\n=== {ds}: multi-anchor candidates ({len(top)}) x {len(SEEDS)} seed(s) ===")
    tm, _ = L.run_many(ds, top.to_dict("records"), data, seeds=SEEDS, cache_dir=CACHE, ids=top.cid.tolist(),
                       offsets=[1000 + int(r) for r in top.row])                       # seed_offset = 1000 + i as in 06_revision_diagnostics
    base = BASE[ds].copy()
    if SMOKE:
        base = base.head(2)
    print(f"\n=== {ds}: baseline-confirmation candidates ({len(base)}) x {len(SEEDS)} seed(s) ===")
    tb, _ = L.run_many(ds, base.to_dict("records"), data, seeds=SEEDS, cache_dir=CACHE, ids=[f"b{int(i)}" for i in base.confirm_id],
                       offsets=[int(i) for i in base.confirm_id])
    RES[ds] = dict(multi=tm, base=tb, base_in=base)

In [ ]:
# ============================================================
# R3.5) Blocks B-D - multi-anchor confirmation, verification rate, gap vs anchor
# ============================================================
ALL, VER = [], []
for ds in DATASETS:
    top = sel[ds]; tm = RES[ds]["multi"]
    d = tm.merge(top[["cid", "anchor", "method", "anchor_index", "anchor_value", "anchor_surrogate", "anchor_pctile", "surrogate_pred",
                      "distance_to_factual", "params_soft_review_budget"]].rename(columns={"cid": "id"}), on="id")
    d["dataset"] = ds
    d["err_val"] = d.f1_val - d.surrogate_pred; d["err_test"] = d.f1_test - d.surrogate_pred
    d["meets_objective_val"] = d.f1_val >= TARGET_MIN; d["meets_objective_test"] = d.f1_test >= TARGET_MIN
    d["close_to_surrogate"] = d.err_val.abs() <= TOL
    ALL.append(d)
D = pd.concat(ALL, ignore_index=True)
save_table(D.drop(columns=["seed_offset"]), "R03_multi_anchor_candidates")

anchor_order = ["p50", "p70", "p85", "p95", "best"]
V = (D.groupby(["dataset", "anchor", "method"]).agg(n=("id", "size"), anchor_pctile=("anchor_pctile", "first"),
                                                    surrogate=("surrogate_pred", "mean"), f1_val=("f1_val", "mean"), f1_test=("f1_test", "mean"),
                                                    err_val=("err_val", "mean"), err_test=("err_test", "mean"),
                                                    verified_objective_val=("meets_objective_val", "mean"),
                                                    verified_objective_test=("meets_objective_test", "mean"),
                                                    close_to_surrogate=("close_to_surrogate", "mean")).reset_index())
V["anchor"] = pd.Categorical(V.anchor, anchor_order, ordered=True); V = V.sort_values(["dataset", "anchor", "method"]).reset_index(drop=True)
display(V.round(3)); save_table(V, "R03_verification_rate_by_anchor")
P = (D.groupby(["dataset", "method"]).agg(n=("id", "size"), surrogate=("surrogate_pred", "mean"), f1_val=("f1_val", "mean"), f1_test=("f1_test", "mean"),
                                          err_val=("err_val", "mean"), err_test=("err_test", "mean"),
                                          verified_objective_val=("meets_objective_val", "mean"),
                                          verified_objective_test=("meets_objective_test", "mean"),
                                          close_to_surrogate=("close_to_surrogate", "mean")).reset_index())
display(P.round(3)); save_table(P, "R03_verification_rate_pooled")

rows = []
for ds, g in D.groupby("dataset"):
    if len(g) < 5:
        continue
    for col in ("anchor_pctile", "anchor_value", "distance_to_factual", "surrogate_pred", "params_soft_review_budget"):
        rho = spearmanr(g[col], g.err_val)[0]
        rows.append(dict(dataset=ds, predictor=col, spearman_with_err_val=rho, n=len(g)))
G = pd.DataFrame(rows); display(G.round(3)); save_table(G, "R03_gap_vs_anchor_predictors")

In [ ]:
# ============================================================
# R3.6) Block E - the baseline-retrieval confirmation, redone with the same pipeline
# ============================================================
rows = []
for ds in DATASETS:
    b = RES[ds]["base_in"].copy(); b["id"] = [f"b{int(i)}" for i in b.confirm_id]
    r = RES[ds]["base"].merge(b[["id", "method", "anchor", "qid", "surrogate_pred"]], on="id")
    r["dataset"] = ds
    rows.append(r)
E = pd.concat(rows, ignore_index=True)
E["err_val"] = E.f1_val - E.surrogate_pred; E["err_test"] = E.f1_test - E.surrogate_pred
save_table(E.drop(columns=["seed_offset"]), "R03_baseline_confirmation_rerun")
Es = (E.groupby(["dataset", "method"]).agg(n=("id", "size"), surrogate=("surrogate_pred", "mean"), f1_val=("f1_val", "mean"), f1_test=("f1_test", "mean"),
                                           auroc_test=("auroc_test", "mean"), err_test=("err_test", "mean"), runtime_sec=("runtime_sec", "mean")).reset_index())
display(Es.round(3)); save_table(Es, "R03_baseline_confirmation_summary")

# compare with the committed file that R00 flagged
old = []
for ds in DATASETS:
    f = DS[ds]["confirm"]
    if f.exists():
        o = pd.read_csv(f)
        pcol = next((c for c in o.columns if "auroc" in c.lower()), None)
        fcol = next((c for c in o.columns if c.lower() in ("f1_test", "f1_final", "f1")), None)
        old.append(dict(dataset=ds, committed_rows=len(o), committed_mean_f1=o[fcol].mean() if fcol else np.nan,
                        committed_mean_auroc=o[pcol].mean() if pcol else np.nan, committed_mean_runtime=o.get("runtime_sec", pd.Series(dtype=float)).mean(),
                        rerun_mean_f1=Es[Es.dataset == ds].f1_test.mean(), rerun_mean_auroc=Es[Es.dataset == ds].auroc_test.mean(),
                        rerun_mean_runtime=Es[Es.dataset == ds].runtime_sec.mean()))
OLD = pd.DataFrame(old); display(OLD.round(3)); save_table(OLD, "R03_baseline_confirmation_old_vs_new")

In [ ]:
# ============================================================
# R3.7) Figures
# ============================================================
col = {"DiCE": "#3b82d4", "CFNOW": "#7c5cd8"}
fig, ax = plt.subplots(1, 2, figsize=WIDE_COL)
for a, ds in zip(ax, DATASETS):
    g = V[V.dataset == ds]
    for m, gg in g.groupby("method"):
        gg = gg.sort_values("anchor")
        a.plot(gg.anchor.astype(str), gg.err_val, "-o", ms=3, color=col.get(m, "k"), label=m)
    a.axhline(0, color="k", lw=.6); a.set_xlabel("Factual anchor (log percentile)"); a.set_ylabel("Re-measured − surrogate (val F1)")
    a.set_title("MVTec" if ds == "mvtec" else "Proditec")
ax[0].legend(frameon=False, fontsize=6)
fig.tight_layout(); save_fig(fig, "R03_gap_by_anchor")

fig, ax = plt.subplots(1, 2, figsize=WIDE_COL)
for a, ds in zip(ax, DATASETS):
    g = D[D.dataset == ds]
    for m, gg in g.groupby("method"):
        a.scatter(gg.surrogate_pred, gg.f1_val, s=12, color=col.get(m, "k"), label=m)
    lo, hi = g[["surrogate_pred", "f1_val"]].min().min() - .03, g[["surrogate_pred", "f1_val"]].max().max() + .03
    a.plot([lo, hi], [lo, hi], "k:", lw=.7); a.axhline(TARGET_MIN, color="r", lw=.5); a.set_xlim(lo, hi); a.set_ylim(lo, hi)
    a.set_xlabel("Surrogate prediction"); a.set_ylabel("Re-measured validation F1"); a.set_title(("MVTec" if ds == "mvtec" else "Proditec") + " (all anchors)")
ax[0].legend(frameon=False, fontsize=6)
fig.tight_layout(); save_fig(fig, "R03_multi_anchor_scatter")

In [ ]:
# ============================================================
# R3.8) Freeze numbers
# ============================================================
merge_numbers("R03_anchors_and_baselines", dict(
    smoke=SMOKE, seeds=list(SEEDS), n_per_cell=N_PER_CELL, target_min=TARGET_MIN,
    primary_anchor=A_tab.set_index("dataset").to_dict(orient="index"),
    verification_pooled=P.round(4).to_dict(orient="records"),
    verification_by_anchor=V.assign(anchor=V.anchor.astype(str)).round(4).to_dict(orient="records"),
    baseline_confirmation=Es.round(4).to_dict(orient="records"),
))